# Mooncake Store 介绍

Mooncake 是一个分布式 KV Cache 传输与存储系统。本节将介绍 Mooncake 的整体架构、核心组件、调用链与数据路径以及部署模式。

本节学习大纲如下：

- Mooncake 的定位与整体架构
- 核心组件：Master、Store、Transfer Engine
- 调用链与数据路径
- 部署模式


## 1. Mooncake 的定位与整体架构

### 1.1 定位

Mooncake 是一个分布式 KV Cache 传输与存储系统，提供基于 Host DRAM 的分布式内存池，支持跨节点 KV Cache 的存储、共享与复用。Mooncake Store 可选开启 SSD offload 扩展容量。

> **注意**：Mooncake Store 管理的是 Host DRAM 内存池，NPU HBM 由推理引擎（vLLM/SGLang）自行管理。在 SGLang HiCache 集成中，Mooncake 作为 L3 分布式存储后端。

### 1.2 整体架构

<img src="./images/mooncake_architecture.svg" alt="Mooncake 架构图" width="80%">

Mooncake 由三个核心组件组成：

- **Master**：全局元数据管理，协调 segment 注册与空间分配
- **Store**：存储层，管理 Host DRAM 内存池，提供 key-value 寻址的 put/get 接口
- **Transfer Engine**：传输引擎，负责跨节点数据搬运，底层调用 HIXL/ADXL



## 2. 核心组件

### 2.1 Master（元数据管理）

Master 是全局协调者，管理两类元数据：

| 元数据 | 说明 |
| --- | --- |
| Segment 注册表 | 记录每台机器贡献了多少 Host DRAM、地址范围、网络位置 |
| Key 索引 | 记录每个 key 对应的数据存在哪个 segment 的哪个 offset |

下面的架构图展示了 Master、Segment 与 put/get 数据流的关系：

<img src="./images/master_segment_flow.svg" alt="Master 与 put/get 流程" width="95%">

**put/get 流程**：Store 向 Master 查询/分配空间，再调 Transfer Engine 传输。完整调用链见 Section 3。

```
Store: "我要存 key='kv_layer_0'，大小 1GB"
  → Master: 根据分配策略选择 segment（默认优先本机）
  → Master: 记录 key → (segment_id, offset) 映射
  → Master: 返回目标地址给 Store（哪个 segment、哪个 offset）
  → Store 调 TE 把数据传到目标地址
```


```
Store: "我要取 key='kv_layer_0'"
  → Master: 查 key 索引，返回 (segment_id, offset)
  → Store: 知道数据在哪台机器的哪段内存，调 TE 取回
```

**Master 的分配策略**：

Master 决定数据存在哪个 segment，默认使用 **LOCAL_FIRST** 策略（优先分配写入方本机的 segment，减少跨机传输）：

| 策略 | 说明 |
| --- | --- |
| LOCAL_FIRST（默认） | 优先本机 segment，本机满了再选远端 |
| RANDOM | 随机选一个有空间的 segment |
| FREE_RATIO_FIRST | 选空闲率最高的 segment（负载均衡） |

### 2.2 Store（存储层）

Store 管理基于 **Host DRAM** 的内存池，核心概念是 **segment**：

- **segment**：一段注册到 Master 的 Host 内存，由各节点贡献（通过 `global_segment_size` 配置大小）
- 所有节点的 segment 汇聚成全局内存池，由 Master 统一调度
- Store 提供 key-value 寻址的零拷贝接口：`batch_put_from`（写入池）、`batch_get_into`（从池读取）

> **注意**：Store 的存储池在 Host DRAM，不在 NPU 显存。Device 显存作为传输的源（put 时）或目标（get 时），但不作为存储池。

### 2.3 Transfer Engine（传输引擎）

Transfer Engine 负责跨节点的数据搬运。一次传输任务由 Batch → Task → Slice 三级组成，支持批量并行和部分失败重试：

| 概念 | 说明 |
| --- | --- |
| Batch | 一次 `submitTransfer` 可提交多个传输请求，并行执行 |
| Task | 每个 TransferRequest 对应一个 Task，独立成功/失败 |
| Slice | Task 可拆分为多个 Slice（分片传输），是实际传输单元 |
| 同步接口 | `batch_put_from` / `batch_get_into` 内部轮询直到全部完成 |
| 错误处理 | 部分失败不中断其他 Task，调用方根据返回值决定重试 |

Transfer Engine 通过 `ascend_direct_transport` 适配器调用 HIXL/ADXL 完成底层传输。


## 3. 调用链与数据路径

<img src="./images/call_chain_and_datapath.svg" alt="调用链与数据路径" width="95%">

如上图所示，`batch_put_from` 的调用链分 7 层：Store Client → Master 分配 → TE → AscendDirectTransport → HIXL → 物理链路。`batch_get_into` 调用链相同，数据流向相反。

> **合流点**：Store 路径（7 层）和 TE 直传路径（2 层，不经 Store/Master）都在 `engine.submitTransfer` 处合流，之后完全相同。这就是前几章学的 HIXL。

Store 的数据路径经过 Host 内存中转：

| 操作 | 数据流向 | 说明 |
| --- | --- | --- |
| `batch_put_from` | D2H（Device → Host segment） | 从 NPU 显存写入 Host 内存池 |
| `batch_get_into` | H2D（Host segment → Device） | 从 Host 内存池读取到 NPU 显存 |

> Store 管"存"（D2H/H2D），PD 直传管"传"（D2D），两者互补。


## 4. 部署模式

Mooncake Store 支持三种部署模式，适用不同场景：

| 模式 | 谁分配内存池 | 谁持有 TE | 缓存持久化 | 适用 TP | 适用场景 |
| --- | --- | --- | --- | --- | --- |
| Embedded | 每个推理进程 | 每个推理进程 | ✗ 退出即丢 | TP=1 | 开发测试 |
| Standalone | mooncake_client | 推理进程 + mooncake_client 各自有 | ✓ 独立进程 | 任意 | 生产环境 |
| Dummy/Real | Real Client | Real Client | ✓ 独立进程 | TP>1 | 一机多卡 |

> 详细部署步骤和架构图见 6.4 Section 2。


## 课后练习

本节介绍了 Mooncake 的架构、核心组件、调用链与数据路径以及部署模式，请根据学习内容完成以下题目进行自测。

1. （判断题）Mooncake 的 Transfer Engine 通过 `ascend_direct_transport` 适配器调用 HIXL/ADXL 实现底层传输。

2. （判断题）Mooncake Store 的内存池基于 NPU HBM（设备显存）构建。

3. （单选题）Standalone 模式下，推理进程应将 `global_segment_size` 设为多少？
    A. -1
    B. 0
    C. 1
    D. 1024

4. （单选题）Mooncake Store 的 `batch_put_from` 操作，数据流向是？
    A. D2D（Device → Device）
    B. D2H（Device → Host DRAM）
    C. H2D（Host DRAM → Device）
    D. H2H（Host → Host）

5. （多选题）Mooncake 的三个核心组件是？
    A. Master
    B. Store
    C. Transfer Engine
    D. Scheduler

6. （多选题）Mooncake 支持的部署模式有？
    A. Embedded 模式
    B. Standalone 模式
    C. Dummy/Real Client 模式
    D. 主从复制模式

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/06.03_answer.txt